In [1]:
import mne.io
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import time
from zoneinfo import ZoneInfo
import os
import re
from scipy.signal import welch
import glob

dir_path = "D:/research assistant/EEG/EEG_data/EEG_VBL/"

In [ ]:
# This cell is to extract the power band column from raw data and output to raw_EEG_VBL folder
raw_path = "D:/research assistant/EEG/EEG_data/EEG_VBL/raw"
for sub_folder in os.listdir(raw_path):
    csv_files = glob.glob(os.path.join(raw_path, sub_folder,"*md.mc.pm.fe.bp.csv"))
    number = re.findall(r'sub_(\d*)', sub_folder)[0]
    if number != "102":
        continue
    clean_dfs = []
    for file_path in csv_files:
        print(f"processing: {file_path}")
        df = pd.read_csv(file_path, header=1)
        clean_df = pd.concat([df.iloc[:, 0], df[["EEG.AF3", "EEG.F7", "EEG.F3", "EEG.FC5", "EEG.T7", "EEG.P7", "EEG.O1", "EEG.O2", "EEG.P8", "EEG.T8", "EEG.FC6", "EEG.F4", "EEG.F8", "EEG.AF4"]]], axis=1)
        new_cols = ["Timestamp", "AF3", "F7", "F3", "FC5", "T7", "P7", "O1", "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"]
        clean_df.columns = new_cols
        clean_df["Timestamp"] = pd.to_numeric(clean_df["Timestamp"], errors="coerce")
        clean_dfs.append(clean_df)
    print("finish processing")
    out_file = os.path.join(dir_path, "raw_EEG_VBL", f"raw_EEG_s{number}.csv")
    if len(clean_dfs) >= 2:
        df_all = pd.concat(clean_dfs, ignore_index=True)
        df_all = df_all.sort_values("Timestamp").reset_index(drop=True)
        
        df_all.to_csv(out_file, index=False, encoding="utf-8-sig")
        print(df_all.head())
        print(df_all.tail())
    else:
        clean_dfs[0].to_csv(out_file, index=False, encoding="utf-8-sig")
    print(f"output sub{number}")

processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_102\102_EPOCX_495456_2025.09.13T11.44.04+08.00.md.mc.pm.fe.bp.csv
finish processing
output sub102


In [ ]:
# This cell is to extract the emotional column from raw data and output to emotional_VBL folder
raw_path = "D:/research assistant/EEG/EEG_data/EEG_VBL/raw"
for sub_folder in os.listdir(raw_path):
    csv_files = glob.glob(os.path.join(raw_path, sub_folder,"*md.mc.pm.fe.bp.csv"))
    number = re.findall(r'sub_(\d*)', sub_folder)[0]
    clean_dfs = []
    for file_path in csv_files:
        print(f"processing: {file_path}")
        df = pd.read_csv(file_path, header=1)
        clean_df = pd.concat([df.iloc[:, 0], df[["PM.Attention.Scaled", "PM.Engagement.Scaled", "PM.Excitement.Scaled", "PM.Stress.Scaled", "PM.Relaxation.Scaled", "PM.Interest.Scaled"]]], axis=1)
        clean_df = clean_df[clean_df["PM.Attention.Scaled"].notna() | clean_df["PM.Engagement.Scaled"].notna() |clean_df["PM.Excitement.Scaled"].notna() |clean_df["PM.Stress.Scaled"].notna() |clean_df["PM.Relaxation.Scaled"].notna() |clean_df["PM.Interest.Scaled"].notna()]
        new_cols = ["Timestamp", "PM.Attention.Scaled", "PM.Engagement.Scaled", "PM.Excitement.Scaled", "PM.Stress.Scaled", "PM.Relaxation.Scaled", "PM.Interest.Scaled"]
        clean_df.columns = new_cols
        clean_df["Timestamp"] = pd.to_numeric(clean_df["Timestamp"], errors="coerce")
        clean_dfs.append(clean_df)
    print("finish processing")
    out_file = os.path.join(dir_path, "emotion_VBL", f"raw_EEG_sub{number}.csv")
    if len(clean_dfs) >= 2:
        df_all = pd.concat(clean_dfs, ignore_index=True)
        df_all = df_all.sort_values("Timestamp").reset_index(drop=True)
        
        df_all.to_csv(out_file, index=False, encoding="utf-8-sig")
        print(df_all.head())
        print(df_all.tail())
    else:
        clean_dfs[0].to_csv(out_file, index=False, encoding="utf-8-sig")
    print(f"output sub{number}")

processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_1\001_EPOCX_279324_2025.03.19T11.32.06+08.00.md.mc.pm.fe.bp.csv
finish processing
output sub1
processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_10\010_EPOCX_282049_2025.04.02T16.13.39+08.00.md.mc.pm.fe.bp.csv
finish processing
output sub10
processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_102\102_EPOCX_495456_2025.09.13T11.44.04+08.00.md.mc.pm.fe.bp.csv
finish processing
output sub102
processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_12\012_EPOCX_283031_2025.04.05T11.47.53+08.00.md.mc.pm.fe.bp.csv
finish processing
output sub12
processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_13\013_EPOCX_283475_2025.04.07T11.42.31+08.00.md.mc.pm.fe.bp.csv
processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_13\3_EPOCX_277049_2025.04.07T11.55.09+08.00.md.mc.pm.fe.bp.csv
processing: D:/research assistant/EEG/EEG_data/EEG_VBL/raw\sub_13\4_EPOCX_277053_2025.04.07T11.58.44+08.00.md.

In [ ]:
# rename the clean data files to obey the naming convention for mne.io.read_raw_fif()
for filename in os.listdir(os.path.join(dir_path,"clean_EEG_VBL")):
    new_filename = filename.replace(".fif", "_raw.fif")
    file_path = os.path.join(dir_path, "clean_EEG_VBL", filename)
    new_file_path = os.path.join(dir_path, "clean_EEG_VBL", new_filename)
    os.rename(file_path, new_file_path)

In [4]:
processed_dir = dir_path + "Processed VBL Files/"
segment_dir = os.path.join(dir_path, "segments")

subject_data = {}
for filename in os.listdir(processed_dir):
    processed_file_path = os.path.join(processed_dir, filename)
    processed_df_1 = pd.read_excel(processed_file_path, sheet_name="process")
    processed_df_2 = pd.read_excel(processed_file_path, sheet_name="process_group")
    processed_df_3 = pd.read_excel(processed_file_path, sheet_name="process_category")
    number = re.findall(r'p0*(\d*)', filename)[0]
    if number != "62":
        continue
    try:
        subject_id = str(number).zfill(2)
        pattern = os.path.join(dir_path, "clean_EEG_VBL", f"EEG_s{subject_id}*_clean_raw.fif")
        matching_files = glob.glob(pattern)
        if not matching_files:
            raise FileNotFoundError(f"No file found for subject {subject_id} with pattern: {pattern}")
        clean_file = mne.io.read_raw_fif(matching_files[0], preload=True)
        data, times = clean_file[:]
        channel_names = clean_file.ch_names
        clean_df = pd.DataFrame(data.T, columns=channel_names)

        raw_file_path = dir_path + "raw_EEG_VBL/raw_EEG_s" + number + ".csv"
        raw_df = pd.read_csv(raw_file_path)
        timestamp = raw_df["Timestamp"]
        clean_df['time'] = timestamp
    except Exception as e:
        print(e)
    segments = []
    output_path = os.path.join(segment_dir, number)    
    for index, row in processed_df_1.iterrows():
        start = row['start_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        end = row['end_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        label_1 = row['process']
        label_2 = processed_df_2.iloc[index]['process_group']
        label_3 = processed_df_3.iloc[index]['process_category']
        print(clean_df)
        segment = clean_df[(clean_df['time'] >= start) & (clean_df['time'] < end + 1)]
        if not segment.empty:
            segment['process'] = label_1
            segment['process_group'] = label_2
            segment['process_category'] = label_3
            segments.append(segment)
            if not os.path.exists(output_path):
                os.makedirs(output_path)
            file_name = row['start_time'].strftime("%Y-%m-%d %H-%M-%S") + "_" + row['end_time'].strftime("%Y-%m-%d %H-%M-%S") + ".xlsx"
            segment.to_excel(os.path.join(output_path, file_name))
    subject_data[number] = segments

Opening raw data file D:/research assistant/EEG/EEG_data/EEG_VBL/clean_EEG_VBL\EEG_s62_clean_raw.fif...
    Range : 0 ... 339517 =      0.000 ...  2652.477 secs
Ready.
Reading 0 ... 339517  =      0.000 ...  2652.477 secs...
              AF3          F7           F3          FC5          T7  \
0       12.547150  -50.690041   -54.866039    -8.079345  -57.133755   
1       11.445131  -44.625626   -59.006004   -23.282810  -62.583462   
2        9.379328  -35.891174   -57.952129   -26.884481  -60.975704   
3        7.444996  -31.479860   -55.436287   -21.335873  -56.907410   
4        6.955080  -35.325333   -56.529003   -17.560038  -57.538559   
...           ...         ...          ...          ...         ...   
339513  -9.236946   98.790298 -1738.526123 -1465.254028 -294.000092   
339514  -8.557892   76.938011 -1810.925293 -1526.155029 -304.526093   
339515   8.090355 -123.225983 -1883.324585 -1587.056030 -315.052063   
339516  24.738602 -323.389984 -1955.723755 -1647.956909 -325.5780

C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process'] = label_1
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process_group'] = label_2
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

              AF3          F7           F3          FC5          T7  \
0       12.547150  -50.690041   -54.866039    -8.079345  -57.133755   
1       11.445131  -44.625626   -59.006004   -23.282810  -62.583462   
2        9.379328  -35.891174   -57.952129   -26.884481  -60.975704   
3        7.444996  -31.479860   -55.436287   -21.335873  -56.907410   
4        6.955080  -35.325333   -56.529003   -17.560038  -57.538559   
...           ...         ...          ...          ...         ...   
339513  -9.236946   98.790298 -1738.526123 -1465.254028 -294.000092   
339514  -8.557892   76.938011 -1810.925293 -1526.155029 -304.526093   
339515   8.090355 -123.225983 -1883.324585 -1587.056030 -315.052063   
339516  24.738602 -323.389984 -1955.723755 -1647.956909 -325.578064   
339517  41.386848 -523.553955 -2028.122925 -1708.857910 -336.104034   

                 P7           O1          O2           P8          T8  \
0        -51.965919   -58.185238 -301.999268  -110.025688  174.748840   
1

C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process'] = label_1
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process_group'] = label_2
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

              AF3          F7           F3          FC5          T7  \
0       12.547150  -50.690041   -54.866039    -8.079345  -57.133755   
1       11.445131  -44.625626   -59.006004   -23.282810  -62.583462   
2        9.379328  -35.891174   -57.952129   -26.884481  -60.975704   
3        7.444996  -31.479860   -55.436287   -21.335873  -56.907410   
4        6.955080  -35.325333   -56.529003   -17.560038  -57.538559   
...           ...         ...          ...          ...         ...   
339513  -9.236946   98.790298 -1738.526123 -1465.254028 -294.000092   
339514  -8.557892   76.938011 -1810.925293 -1526.155029 -304.526093   
339515   8.090355 -123.225983 -1883.324585 -1587.056030 -315.052063   
339516  24.738602 -323.389984 -1955.723755 -1647.956909 -325.578064   
339517  41.386848 -523.553955 -2028.122925 -1708.857910 -336.104034   

                 P7           O1          O2           P8          T8  \
0        -51.965919   -58.185238 -301.999268  -110.025688  174.748840   
1

...

C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process'] = label_1
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process_group'] = label_2
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

              AF3          F7           F3          FC5          T7  \
0       12.547150  -50.690041   -54.866039    -8.079345  -57.133755   
1       11.445131  -44.625626   -59.006004   -23.282810  -62.583462   
2        9.379328  -35.891174   -57.952129   -26.884481  -60.975704   
3        7.444996  -31.479860   -55.436287   -21.335873  -56.907410   
4        6.955080  -35.325333   -56.529003   -17.560038  -57.538559   
...           ...         ...          ...          ...         ...   
339513  -9.236946   98.790298 -1738.526123 -1465.254028 -294.000092   
339514  -8.557892   76.938011 -1810.925293 -1526.155029 -304.526093   
339515   8.090355 -123.225983 -1883.324585 -1587.056030 -315.052063   
339516  24.738602 -323.389984 -1955.723755 -1647.956909 -325.578064   
339517  41.386848 -523.553955 -2028.122925 -1708.857910 -336.104034   

                 P7           O1          O2           P8          T8  \
0        -51.965919   -58.185238 -301.999268  -110.025688  174.748840   
1

C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process'] = label_1
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process_group'] = label_2
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

              AF3          F7           F3          FC5          T7  \
0       12.547150  -50.690041   -54.866039    -8.079345  -57.133755   
1       11.445131  -44.625626   -59.006004   -23.282810  -62.583462   
2        9.379328  -35.891174   -57.952129   -26.884481  -60.975704   
3        7.444996  -31.479860   -55.436287   -21.335873  -56.907410   
4        6.955080  -35.325333   -56.529003   -17.560038  -57.538559   
...           ...         ...          ...          ...         ...   
339513  -9.236946   98.790298 -1738.526123 -1465.254028 -294.000092   
339514  -8.557892   76.938011 -1810.925293 -1526.155029 -304.526093   
339515   8.090355 -123.225983 -1883.324585 -1587.056030 -315.052063   
339516  24.738602 -323.389984 -1955.723755 -1647.956909 -325.578064   
339517  41.386848 -523.553955 -2028.122925 -1708.857910 -336.104034   

                 P7           O1          O2           P8          T8  \
0        -51.965919   -58.185238 -301.999268  -110.025688  174.748840   
1

C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process'] = label_1
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  segment['process_group'] = label_2
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_33240\2701164002.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

In [12]:
processed_dir = dir_path + "Processed VBL Files/"
segment_dir = os.path.join(dir_path, "segments_emotion")

subject_data = {}
for filename in os.listdir(processed_dir):
    processed_file_path = os.path.join(processed_dir, filename)
    processed_df_1 = pd.read_excel(processed_file_path, sheet_name="process_category")
    processed_df_2 = pd.read_excel(processed_file_path, sheet_name="process")
    processed_df_3 = pd.read_excel(processed_file_path, sheet_name="process_group")
    number = re.findall(r'p0*(\d*)', filename)[0]
    print(f"processing subject {number}")
    try:
        raw_file_path = dir_path + "emotion_VBL/raw_EEG_sub" + number + ".csv"
        raw_df = pd.read_csv(raw_file_path)
    except Exception as e:
        print(e)
    segments = []
    output_path = os.path.join(segment_dir, number)
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    for index, row in processed_df_1.iterrows():
        start = row['start_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        end = row['end_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        label_1 = row['process_category']
        label_2 = processed_df_2.iloc[index]['process']
        label_3 = processed_df_3.iloc[index]['process_group']
        segment = raw_df[(raw_df['Timestamp'] >= start) & (raw_df['Timestamp'] <= end)]
        segment['process_category'] = label_1
        segment['process'] = label_2
        segment['process_group'] = label_3
        if not segment.empty:
            segments.append(segment)
            file_name = row['start_time'].strftime("%Y-%m-%d %H-%M-%S") + "_" + row['end_time'].strftime("%Y-%m-%d %H-%M-%S") + ".xlsx"
            segment.to_excel(os.path.join(output_path, file_name))
    subject_data[number] = segments

processing subject 1
processing subject 2
processing subject 3
processing subject 4
processing subject 6
processing subject 7
processing subject 9
processing subject 11
[Errno 2] No such file or directory: 'D:/research assistant/EEG/EEG_data/EEG_VBL/emotion_VBL/raw_EEG_sub11.csv'
processing subject 12
processing subject 13
processing subject 14
processing subject 15
processing subject 16
processing subject 17
processing subject 18
processing subject 19
[Errno 2] No such file or directory: 'D:/research assistant/EEG/EEG_data/EEG_VBL/emotion_VBL/raw_EEG_sub19.csv'
processing subject 20
processing subject 21
processing subject 22
processing subject 23
processing subject 25
processing subject 26
[Errno 2] No such file or directory: 'D:/research assistant/EEG/EEG_data/EEG_VBL/emotion_VBL/raw_EEG_sub26.csv'
processing subject 27
processing subject 28
processing subject 29
processing subject 30
processing subject 32
processing subject 38
processing subject 43
processing subject 62
processing 

In [16]:
clean_numbers = []
for file in os.listdir(os.path.join(dir_path,"clean_EEG_VBL")):
    clean_numbers.append(re.findall(r's0*(\d*)', file)[0])

In [18]:
list(set(clean_numbers) - set(os.listdir(os.path.join(dir_path, "segments"))))

['102', '5', '10']